# Modelo Espacio-Temporal: CNN-LSTM (Modelo 3)

Este cuaderno implementa el entrenamiento y la evaluación de la arquitectura combinada **CNN-LSTM** para la estimación del tiempo transcurrido desde el contacto del rastro térmico.

### Características de la arquitectura:
1. **Backbone Visual (CNN 2D):** Utiliza la rama ligera **Lite-DSTFS** (~750K parámetros) heredada de MTDE-Net para extraer características espaciales del decaimiento térmico cuadro por cuadro.
2. **Capa Recurrente (LSTM):** Modela la dinámica de enfriamiento temporal sobre secuencias de características visuales extraídas por la CNN.
3. **Ventanas Móviles (Sliding Windows):** Genera ventanas deslizantes de tamaño fijo (por defecto, $L=5$ fotogramas consecutivos) garantizando la consistencia física de que todos los cuadros dentro de una ventana pertenezcan a la misma secuencia de contacto (`sequence_id`).

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.models.cnn_lstm import ThermalCNNLSTM
from src.loaders.cnn_lstm_loader import ThermalSequenceDataset
from src.utils import SqrtScaledMSELoss, eval_cnn_lstm_metrics, build_subject_split_plan, subject_summary, sequence_ids_for_subjects, plot_learning_curves, plot_prediction_calibration, plot_error_by_time_window

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Configuración General (Hiperparámetros)

In [3]:
CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    "batch_size": 8,           # Un batch size menor es recomendado para datos secuenciales pesados
    "lr": 0.0003,              # AdamW tasa de aprendizaje adaptativa
    "weight_decay": 0.001,
    "min_time_s": 0.0,
    "seq_len": 5,              # Tamaño de la ventana deslizante (5 cuadros consecutivos)
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "time_scale": 30.0,
}

### 3. Partición de Secuencias y Carga de Datasets (Sin Data Leakage)

Para garantizar la validez científica del experimento, dividimos los datos **estrictamente por secuencias físicas de contacto** (sequence_id) antes de generar las ventanas móviles.

In [4]:
dev = CONFIG["device"]
metadata_path = "../processed_data/metadata_train.csv"
df = pd.read_csv(metadata_path)

print("Resumen de sujetos en metadata:")
print(subject_summary(df))

plan = build_subject_split_plan(
    df,
    n_splits=5,
    seed=42,
    test_subjects=["tania"],
    reserve_incomplete_for_test=True,
)

print("\nPlan de partición:")
print(f"Sujetos de Test ({plan.num_test_subjects}): {plan.test_subjects} (Muestras: {plan.num_test_samples})")
print(f"Sujetos de Train/Val ({plan.num_trainval_subjects}): {plan.trainval_subjects} (Muestras: {plan.num_trainval_samples})")
print(f"Número de folds para CV: {len(plan.folds)}")
for f in plan.folds:
    print(f"  Fold {f.fold}: Train subjects {f.train_subjects}, Val subjects {f.val_subjects}")

Resumen de sujetos en metadata:
        name  n_sequences  n_samples      surface  gender  is_complete
0     alonso            3        141        glass    male        False
1     angelo            6        281  glass, wood    male         True
2    eduardo            1         47         wood    male        False
3    esteban            6        276  glass, wood    male         True
4   fabricio            6        303  glass, wood    male         True
5    johamin            2         93         wood    male        False
6  juandiego            6        355  glass, wood    male         True
7     matias            6        282  glass, wood    male         True
8     renato            3        141         wood    male        False
9      tania            6        278  glass, wood  female         True

Plan de partición:
Sujetos de Test (5): ['alonso', 'eduardo', 'johamin', 'renato', 'tania'] (Muestras: 700)
Sujetos de Train/Val (5): ['angelo', 'esteban', 'fabricio', 'juandiego', 'mati

### 4. Inicialización del Modelo, Pérdida y Optimización

In [5]:
model = ThermalCNNLSTM().to(dev)
print(f"Modelo CNN-LSTM inicializado con {sum(p.numel() for p in model.parameters()):,} parámetros")

Modelo CNN-LSTM inicializado con 1,445,805 parámetros


--- 
## FASE A: Entrenamiento del Modelo Baseline (Sin Tunear)

Esta sección entrena el modelo CNN-LSTM espacio-temporal con los hiperparámetros por defecto.

In [ ]:
BASELINE_CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    "batch_size": 8,
    "lr": 0.0003,
    "weight_decay": 0.001,
}

set_seed(42)

fold_metrics_base = []
histories_base = []

print("=" * 60)
print("FASE A: VALIDACIÓN CRUZADA POR SUJETO (BASELINE)")
print("=" * 60)

for fold in plan.folds:
    print(f"\nEntrenando Fold {fold.fold}/{len(plan.folds)} <<<")
    print(f"Train subjects: {fold.train_subjects}")
    print(f"Val subjects: {fold.val_subjects}")
    
    train_seq_ids = sequence_ids_for_subjects(df, fold.train_subjects)
    val_seq_ids = sequence_ids_for_subjects(df, fold.val_subjects)
    
    train_ds = ThermalSequenceDataset(
        metadata_csv=metadata_path,
        is_train=True,
        min_time_s=CONFIG["min_time_s"],
        seq_len=CONFIG["seq_len"],
        sequence_ids=train_seq_ids
    )
    val_ds = ThermalSequenceDataset(
        metadata_csv=metadata_path,
        is_train=False,
        min_time_s=CONFIG["min_time_s"],
        seq_len=CONFIG["seq_len"],
        sequence_ids=val_seq_ids
    )
    
    train_loader = DataLoader(train_ds, batch_size=BASELINE_CONFIG["batch_size"], shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BASELINE_CONFIG["batch_size"])
    train_eval_loader = DataLoader(train_ds, batch_size=BASELINE_CONFIG["batch_size"])
    
    model = ThermalCNNLSTM().to(dev)
    crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])
    opt = torch.optim.AdamW(model.parameters(), lr=BASELINE_CONFIG["lr"], weight_decay=BASELINE_CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[25, 50, 70], gamma=0.8)
    
    best_val_mae = float("inf")
    no_imp = 0
    best_metrics = None
    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}
    
    for ep in range(1, BASELINE_CONFIG["epochs"] + 1):
        model.train()
        train_loss, n = 0.0, 0
        for x_seq, y in train_loader:
            x_seq, y = x_seq.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x_seq), y)
            loss.backward()
            opt.step()
            train_loss += loss.item() * x_seq.size(0)
            n += x_seq.size(0)
            
        scheduler.step()
        
        model.eval()
        val_loss, val_n = 0.0, 0
        with torch.no_grad():
            for val_x_seq, val_y in val_loader:
                val_x_seq, val_y = val_x_seq.to(dev), val_y.to(dev)
                loss = crit(model(val_x_seq), val_y)
                val_loss += loss.item() * val_x_seq.size(0)
                val_n += val_x_seq.size(0)
        val_loss /= val_n
        
        val_m = eval_cnn_lstm_metrics(model, val_loader, dev)
        train_m = eval_cnn_lstm_metrics(model, train_eval_loader, dev)
        
        v_mae = val_m["mae"]
        is_best = v_mae < best_val_mae - CONFIG["min_delta"]
        if is_best:
            best_val_mae, no_imp = v_mae, 0
            best_metrics = val_m
            torch.save(model.state_dict(), f"../CNNLSTM_baseline_fold{fold.fold}.pt")
        else:
            no_imp += 1
            
        history["train_loss"].append(train_loss / n)
        history["val_loss"].append(val_loss)
        history["train_mae"].append(train_m["mae"])
        history["val_mae"].append(v_mae)
        
        if no_imp >= CONFIG["patience"]:
            break
            
    print(f"Fold {fold.fold} finalizado. Mejor Val MAE: {best_val_mae:.2f}s")
    fold_metrics_base.append(best_metrics)
    histories_base.append(history)

# Calcular promedio y desviación estándar de validación de los folds
df_folds_base = pd.DataFrame(fold_metrics_base)
cv_mean_base = df_folds_base.mean().to_dict()
cv_std_base = df_folds_base.std().to_dict()

print("\n" + "=" * 60)
print("FASE A: ENTRENAMIENTO FINAL EN ALL TRAINVAL Y EVAL EN TEST (BASELINE)")
print("=" * 60)

trainval_seq_ids = sequence_ids_for_subjects(df, plan.trainval_subjects)
test_seq_ids = sequence_ids_for_subjects(df, plan.test_subjects)

trainval_ds = ThermalSequenceDataset(metadata_csv=metadata_path, is_train=True, min_time_s=CONFIG["min_time_s"], seq_len=CONFIG["seq_len"], sequence_ids=trainval_seq_ids)
test_ds = ThermalSequenceDataset(metadata_csv=metadata_path, is_train=False, min_time_s=CONFIG["min_time_s"], seq_len=CONFIG["seq_len"], sequence_ids=test_seq_ids)

trainval_loader = DataLoader(trainval_ds, batch_size=BASELINE_CONFIG["batch_size"], shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=BASELINE_CONFIG["batch_size"])

model_base = ThermalCNNLSTM().to(dev)
crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])
opt = torch.optim.AdamW(model_base.parameters(), lr=BASELINE_CONFIG["lr"], weight_decay=BASELINE_CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[25, 50, 70], gamma=0.8)

avg_epochs_base = int(np.mean([len(h["train_loss"]) - CONFIG["patience"] for h in histories_base]))
avg_epochs_base = max(10, avg_epochs_base)
print(f"Entrenando modelo final por {avg_epochs_base} épocas...")

for ep in range(1, avg_epochs_base + 1):
    model_base.train()
    train_loss, n = 0.0, 0
    for x_seq, y in trainval_loader:
        x_seq, y = x_seq.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model_base(x_seq), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_seq.size(0)
        n += x_seq.size(0)
    scheduler.step()

torch.save(model_base.state_dict(), "../CNNLSTM_baseline.pt")
baseline_metrics = eval_cnn_lstm_metrics(model_base, test_loader, dev)
print(f"Modelo Baseline Final evaluado en Test Set. MAE: {baseline_metrics['mae']:.2f}s | RMSE: {baseline_metrics['rmse']:.2f}s")

FASE A: VALIDACIÓN CRUZADA POR SUJETO (BASELINE)

Entrenando Fold 1/5 <<<
Train subjects: ['angelo', 'esteban', 'fabricio', 'juandiego']
Val subjects: ['matias']


--- 
## FASE B: Entrenamiento del Modelo Optimizado (Optuna Champion)

Esta sección entrena el modelo con los mejores hiperparámetros optimizados sugeridos por Optuna.

In [ ]:
# NOTA: Edita este diccionario con los mejores parámetros una vez completes tu búsqueda Optuna
OPTIMIZED_CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    'lr': 0.0009006498540113824, 
    'weight_decay': 0.00015846787710903097, 
    'batch_size': 16, 
    'dropout': 0.05680106209474432
}

set_seed(42)

fold_metrics_opt = []
histories_opt = []

print("=" * 60)
print("FASE B: VALIDACIÓN CRUZADA POR SUJETO (OPTIMIZADO)")
print("=" * 60)

for fold in plan.folds:
    print(f"\nEntrenando Fold {fold.fold}/{len(plan.folds)} <<<")
    print(f"Train subjects: {fold.train_subjects}")
    print(f"Val subjects: {fold.val_subjects}")
    
    train_seq_ids = sequence_ids_for_subjects(df, fold.train_subjects)
    val_seq_ids = sequence_ids_for_subjects(df, fold.val_subjects)
    
    train_ds = ThermalSequenceDataset(metadata_csv=metadata_path, is_train=True, min_time_s=CONFIG["min_time_s"], seq_len=CONFIG["seq_len"], sequence_ids=train_seq_ids)
    val_ds = ThermalSequenceDataset(metadata_csv=metadata_path, is_train=False, min_time_s=CONFIG["min_time_s"], seq_len=CONFIG["seq_len"], sequence_ids=val_seq_ids)
    
    train_loader = DataLoader(train_ds, batch_size=OPTIMIZED_CONFIG["batch_size"], shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=OPTIMIZED_CONFIG["batch_size"])
    train_eval_loader = DataLoader(train_ds, batch_size=OPTIMIZED_CONFIG["batch_size"])
    
    model = ThermalCNNLSTM(dropout=OPTIMIZED_CONFIG["dropout"]).to(dev)
    crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])
    opt = torch.optim.AdamW(model.parameters(), lr=OPTIMIZED_CONFIG["lr"], weight_decay=OPTIMIZED_CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[25, 50, 70], gamma=0.8)
    
    best_val_mae = float("inf")
    no_imp = 0
    best_metrics = None
    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}
    
    for ep in range(1, OPTIMIZED_CONFIG["epochs"] + 1):
        model.train()
        train_loss, n = 0.0, 0
        for x_seq, y in train_loader:
            x_seq, y = x_seq.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x_seq), y)
            loss.backward()
            opt.step()
            train_loss += loss.item() * x_seq.size(0)
            n += x_seq.size(0)
            
        scheduler.step()
        
        model.eval()
        val_loss, val_n = 0.0, 0
        with torch.no_grad():
            for val_x_seq, val_y in val_loader:
                val_x_seq, val_y = val_x_seq.to(dev), val_y.to(dev)
                loss = crit(model(val_x_seq), val_y)
                val_loss += loss.item() * val_x_seq.size(0)
                val_n += val_x_seq.size(0)
        val_loss /= val_n
        
        val_m = eval_cnn_lstm_metrics(model, val_loader, dev)
        train_m = eval_cnn_lstm_metrics(model, train_eval_loader, dev)
        
        v_mae = val_m["mae"]
        is_best = v_mae < best_val_mae - CONFIG["min_delta"]
        if is_best:
            best_val_mae, no_imp = v_mae, 0
            best_metrics = val_m
            torch.save(model.state_dict(), f"../CNNLSTM_optimized_fold{fold.fold}.pt")
        else:
            no_imp += 1
            
        history["train_loss"].append(train_loss / n)
        history["val_loss"].append(val_loss)
        history["train_mae"].append(train_m["mae"])
        history["val_mae"].append(v_mae)
        
        if no_imp >= CONFIG["patience"]:
            break
            
    print(f"Fold {fold.fold} finalizado. Mejor Val MAE: {best_val_mae:.2f}s")
    fold_metrics_opt.append(best_metrics)
    histories_opt.append(history)

# Calcular promedio y desviación estándar de validación de los folds
df_folds_opt = pd.DataFrame(fold_metrics_opt)
cv_mean_opt = df_folds_opt.mean().to_dict()
cv_std_opt = df_folds_opt.std().to_dict()

print("\n" + "=" * 60)
print("FASE B: ENTRENAMIENTO FINAL EN ALL TRAINVAL Y EVAL EN TEST (OPTIMIZADO)")
print("=" * 60)

trainval_seq_ids = sequence_ids_for_subjects(df, plan.trainval_subjects)
test_seq_ids = sequence_ids_for_subjects(df, plan.test_subjects)
trainval_ds = ThermalSequenceDataset(metadata_csv=metadata_path, is_train=True, min_time_s=CONFIG["min_time_s"], seq_len=CONFIG["seq_len"], sequence_ids=trainval_seq_ids)
test_ds = ThermalSequenceDataset(metadata_csv=metadata_path, is_train=False, min_time_s=CONFIG["min_time_s"], seq_len=CONFIG["seq_len"], sequence_ids=test_seq_ids)
trainval_loader = DataLoader(trainval_ds, batch_size=OPTIMIZED_CONFIG["batch_size"], shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=OPTIMIZED_CONFIG["batch_size"])

model_opt = ThermalCNNLSTM(dropout=OPTIMIZED_CONFIG["dropout"]).to(dev)
crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])
opt = torch.optim.AdamW(model_opt.parameters(), lr=OPTIMIZED_CONFIG["lr"], weight_decay=OPTIMIZED_CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[25, 50, 70], gamma=0.8)

avg_epochs_opt = int(np.mean([len(h["train_loss"]) - CONFIG["patience"] for h in histories_opt]))
avg_epochs_opt = max(10, avg_epochs_opt)
print(f"Entrenando modelo final por {avg_epochs_opt} épocas...")

for ep in range(1, avg_epochs_opt + 1):
    model_opt.train()
    train_loss, n = 0.0, 0
    for x_seq, y in trainval_loader:
        x_seq, y = x_seq.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model_opt(x_seq), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_seq.size(0)
        n += x_seq.size(0)
    scheduler.step()

torch.save(model_opt.state_dict(), "../CNNLSTM_final.pt")
optimized_metrics = eval_cnn_lstm_metrics(model_opt, test_loader, dev)
print(f"Modelo Optimizado Final evaluado en Test Set. MAE: {optimized_metrics['mae']:.2f}s | RMSE: {optimized_metrics['rmse']:.2f}s")

--- 
## 3. Tabla Comparativa de Resultados Finales

Esta celda autogenera una tabla markdown limpia comparando ambos escenarios listos para exportar.

In [ ]:
comparative_data = {
    "Métrica": [
        "MAE CV (Validación)",
        "RMSE CV (Validación)",
        "R² CV (Validación)",
        "MAE Test (Generalización)",
        "RMSE Test (Generalización)",
        "R² Test (Generalización)",
        "MAPE Test (Error %)",
        "Acc@60s Test",
        "Acc@120s Test"
    ],
    "Baseline (Default)": [
        f"{cv_mean_base['mae']:.2f} ± {cv_std_base['mae']:.2f} s",
        f"{cv_mean_base['rmse']:.2f} ± {cv_std_base['rmse']:.2f} s",
        f"{cv_mean_base['r2']:.4f} ± {cv_std_base['r2']:.4f}",
        f"{baseline_metrics['mae']:.2f} s",
        f"{baseline_metrics['rmse']:.2f} s",
        f"{baseline_metrics['r2']:.4f}",
        f"{baseline_metrics['mape']:.2f} %",
        f"{baseline_metrics['acc60']:.2f} %",
        f"{baseline_metrics['acc120']:.2f} %"
    ],
    "Optimizado (Optuna)": [
        f"{cv_mean_opt['mae']:.2f} ± {cv_std_opt['mae']:.2f} s",
        f"{cv_mean_opt['rmse']:.2f} ± {cv_std_opt['rmse']:.2f} s",
        f"{cv_mean_opt['r2']:.4f} ± {cv_std_opt['r2']:.4f}",
        f"{optimized_metrics['mae']:.2f} s",
        f"{optimized_metrics['rmse']:.2f} s",
        f"{optimized_metrics['r2']:.4f}",
        f"{optimized_metrics['mape']:.2f} %",
        f"{optimized_metrics['acc60']:.2f} %",
        f"{optimized_metrics['acc120']:.2f} %"
    ]
}

df_compare = pd.DataFrame(comparative_data)
from IPython.display import display, Markdown
display(Markdown("### Tabla Comparativa de CNN-LSTM para Tesis"))
df_compare

--- 
## 4. Visualización de Resultados Experimentales y Curvas de Aprendizaje

In [ ]:
import numpy as np

# Cargar el modelo final entrenado para graficar y evaluar predicciones en Test
model_opt.load_state_dict(torch.load("../CNNLSTM_final.pt"))
model_opt.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x_seq, y in test_loader:
        x_seq, y = x_seq.to(dev), y.to(dev)
        pred = model_opt(x_seq)
        # Desescalar predicciones y reales
        y_true.extend((y.cpu().numpy() * CONFIG["time_scale"]).tolist())
        y_pred.extend((pred.cpu().numpy() * CONFIG["time_scale"]).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# 1. Curvas de aprendizaje del último fold de validación cruzada (Optimizado)
plot_learning_curves(histories_opt[-1])

# 2. Gráfico de Calibración de Predicciones en el Test Set
plot_prediction_calibration(y_true, y_pred, model_name="CNN-LSTM (Test Set - Optimizado)")

# 3. Distribución del error por ventana temporal en el Test Set
plot_error_by_time_window(y_true, y_pred, model_name="CNN-LSTM (Test Set - Optimizado)")